# Exploratory Data Analysis (EDA) of S&P 500 Stock Data

## Introduction
This notebook performs **Exploratory Data Analysis (EDA)** on the **S&P 500 stock data** dataset.  
The goal is to understand the structure of the dataset, inspect trends, check for missing values, study stock price behavior, and generate insights using visualizations.

## Objectives
- Load and inspect the dataset
- Understand the columns and data types
- Check for missing values and duplicates
- Explore the distribution of stock prices and trading volume
- Analyze stock trends over time
- Compute daily returns and volatility
- Study correlations among selected companies
- Summarize key insights

## Dataset
The dataset contains historical stock data for companies in the S&P 500 index.  
Each row represents a stock’s daily trading data.

## Expected Columns
- **date**: Trading date
- **open**: Opening stock price
- **high**: Highest stock price of the day
- **low**: Lowest stock price of the day
- **close**: Closing stock price
- **volume**: Number of shares traded
- **Name**: Stock ticker/company symbol

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Plot settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

## 1. Load the Dataset
In Kaggle, dataset paths may vary slightly depending on the folder name.  
The following code automatically finds the main CSV file.

In [ ]:
# # Automatically locate the dataset file inside /kaggle/input
# csv_path = "all_stocks_5yr.csv"
# csv_files = list(input_path.rglob("all_stocks_5yr.csv"))

# if not csv_files:
#     raise FileNotFoundError("Could not find 'all_stocks_5yr.csv' inside /kaggle/input")

# csv_path = csv_files[0]
# print("Dataset found at:", csv_path)

# # Load dataset
# df = pd.read_csv(csv_path)

# # Display first 5 rows
# df.head()

import pandas as pd

csv_path = "all_stocks_5yr.csv"
df = pd.read_csv(csv_path)

df.head()

## 2. Basic Overview
We first inspect the dataset’s shape, columns, data types, and a general summary.

In [ ]:
# Shape of dataset
print("Shape of dataset:", df.shape)

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types
print("\nData types:")
print(df.dtypes)

# General info
print("\nDataset Info:")
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe().T

## 3. Data Cleaning and Preprocessing
We will:
- Convert the `date` column to datetime format
- Check for missing values
- Check for duplicate rows
- Sort values by stock name and date

In [ ]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort data
df = df.sort_values(["Name", "date"]).reset_index(drop=True)

# Missing values
missing_values = df.isnull().sum()

# Duplicates
duplicate_rows = df.duplicated().sum()

print("Missing values:\n")
print(missing_values)

print("\nNumber of duplicate rows:", duplicate_rows)

In [ ]:
# Remove duplicates if any
df = df.drop_duplicates()

# Confirm shape after cleaning
print("Shape after cleaning:", df.shape)

# Display cleaned data
df.head()

## 4. Dataset-Level Exploration
Let us explore:
- Total number of companies
- Date range covered
- Number of observations per company

In [ ]:
num_companies = df["Name"].nunique()
date_min = df["date"].min()
date_max = df["date"].max()

print("Number of unique companies:", num_companies)
print("Date range:", date_min.date(), "to", date_max.date())
print("Total observations:", len(df))

In [ ]:
# Observations per company
company_counts = df["Name"].value_counts()

print("Top 10 companies by number of records:")
print(company_counts.head(10))

## 5. Univariate Analysis
We now study the distribution of:
- Closing prices
- Opening prices
- Trading volume

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df["close"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Closing Prices")
axes[0].set_xlabel("Close Price")

sns.histplot(np.log1p(df["volume"]), bins=50, kde=True, ax=axes[1])
axes[1].set_title("Distribution of Log(1 + Volume)")
axes[1].set_xlabel("log(1 + Volume)")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for price columns
price_cols = ["open", "high", "low", "close"]

plt.figure(figsize=(12, 6))
sns.boxplot(data=df[price_cols])
plt.title("Boxplot of Price Columns")
plt.ylabel("Price")
plt.show()

## 6. Company-Wise Summary Statistics
We will compute summary statistics for each company:
- Average closing price
- Standard deviation of closing price
- Average trading volume
- Number of trading days

In [ ]:
company_summary = df.groupby("Name").agg(
    trading_days=("date", "count"),
    avg_open=("open", "mean"),
    avg_close=("close", "mean"),
    std_close=("close", "std"),
    avg_volume=("volume", "mean"),
    min_close=("close", "min"),
    max_close=("close", "max")
).sort_values("avg_close", ascending=False)

company_summary.head(10)

In [ ]:
# Top 10 companies by average closing price
top_avg_close = company_summary.head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_avg_close.index, y=top_avg_close["avg_close"])
plt.title("Top 10 Companies by Average Closing Price")
plt.xlabel("Company")
plt.ylabel("Average Close Price")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Top 10 companies by average trading volume
top_volume = company_summary.sort_values("avg_volume", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_volume.index, y=top_volume["avg_volume"])
plt.title("Top 10 Companies by Average Trading Volume")
plt.xlabel("Company")
plt.ylabel("Average Volume")
plt.xticks(rotation=45)
plt.show()

## 7. Time Series Analysis of Selected Stocks
Visualizing all 500 companies together would be too crowded.  
So we select a few well-known companies and analyze their closing prices over time.

In [ ]:
# Preferred companies to visualize
preferred_stocks = ["AAPL", "AMZN", "GOOGL", "MSFT", "META"]

# Keep only those present in dataset
selected_stocks = [stock for stock in preferred_stocks if stock in df["Name"].unique()]

# If some are not available, choose first 4 available companies from dataset
if len(selected_stocks) < 4:
    selected_stocks = list(df["Name"].unique()[:4])

print("Selected stocks:", selected_stocks)

In [ ]:
plt.figure(figsize=(14, 7))

for stock in selected_stocks:
    stock_data = df[df["Name"] == stock]
    plt.plot(stock_data["date"], stock_data["close"], label=stock)

plt.title("Closing Price Trends of Selected Stocks")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.show()

## 8. Moving Average Analysis
Moving averages help smooth short-term fluctuations and show longer-term trends.
We will compute:
- 20-day moving average
- 50-day moving average

In [ ]:
# Choose one stock for moving average analysis
stock_for_ma = selected_stocks[0]
stock_df = df[df["Name"] == stock_for_ma].copy()

stock_df["MA_20"] = stock_df["close"].rolling(window=20).mean()
stock_df["MA_50"] = stock_df["close"].rolling(window=50).mean()

plt.figure(figsize=(14, 7))
plt.plot(stock_df["date"], stock_df["close"], label="Close Price")
plt.plot(stock_df["date"], stock_df["MA_20"], label="20-Day MA")
plt.plot(stock_df["date"], stock_df["MA_50"], label="50-Day MA")

plt.title(f"{stock_for_ma} Closing Price with Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

## 9. Daily Returns Analysis
Daily return shows the percentage change in closing price from one day to the next.

Formula:

\[
\text{Daily Return} = \frac{\text{Close}_t - \text{Close}_{t-1}}{\text{Close}_{t-1}}
\]

In [ ]:
# Calculate daily return for each company
df["daily_return"] = df.groupby("Name")["close"].pct_change()

# Display sample
df[["date", "Name", "close", "daily_return"]].head(10)

In [ ]:
# Plot return distributions for selected stocks
plt.figure(figsize=(14, 7))

for stock in selected_stocks:
    stock_returns = df[df["Name"] == stock]["daily_return"].dropna()
    sns.kdeplot(stock_returns, label=stock, fill=False)

plt.title("Distribution of Daily Returns for Selected Stocks")
plt.xlabel("Daily Return")
plt.ylabel("Density")
plt.legend()
plt.show()

## 10. Volatility Analysis
Volatility measures how much a stock’s returns fluctuate over time.  
A common measure is the standard deviation of daily returns.

In [ ]:
volatility = df.groupby("Name")["daily_return"].std().sort_values(ascending=False)

volatility_df = volatility.head(15).reset_index()
volatility_df.columns = ["Name", "Volatility"]

volatility_df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=volatility_df, x="Name", y="Volatility")
plt.title("Top 15 Most Volatile Stocks")
plt.xlabel("Company")
plt.ylabel("Standard Deviation of Daily Returns")
plt.xticks(rotation=45)
plt.show()

## 11. Correlation Analysis
To study relationships between stocks, we pivot the data so that:
- Rows represent dates
- Columns represent stock symbols
- Values represent closing prices

Then we calculate correlations based on daily returns.

In [ ]:
# Use top 10 companies by average volume for correlation study
top_corr_stocks = top_volume.index.tolist()

corr_data = df[df["Name"].isin(top_corr_stocks)].pivot(index="date", columns="Name", values="close")
corr_returns = corr_data.pct_change().dropna()

corr_matrix = corr_returns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Daily Returns (Top Volume Stocks)")
plt.show()

## 12. Monthly Trend Analysis
To better understand long-term trends, we can aggregate closing prices by month.

In [ ]:
monthly_df = df.copy()
monthly_df["year_month"] = monthly_df["date"].dt.to_period("M")

monthly_close = (
    monthly_df[monthly_df["Name"].isin(selected_stocks)]
    .groupby(["year_month", "Name"])["close"]
    .mean()
    .reset_index()
)

monthly_close["year_month"] = monthly_close["year_month"].astype(str)

plt.figure(figsize=(15, 7))
sns.lineplot(data=monthly_close, x="year_month", y="close", hue="Name", marker="o")
plt.title("Monthly Average Closing Price of Selected Stocks")
plt.xlabel("Year-Month")
plt.ylabel("Average Close Price")
plt.xticks(rotation=90)
plt.show()

## 13. Key Findings
Based on the EDA, we can observe the following:

1. The dataset contains stock market information for many companies in the S&P 500 index.
2. The data spans multiple years of daily trading activity.
3. Closing prices and trading volumes vary greatly across companies.
4. Some companies show strong upward trends, while others fluctuate more heavily.
5. Daily returns are generally centered around zero, but extreme values indicate occasional large movements.
6. Volatility differs across companies, meaning some stocks are riskier than others.
7. Correlation analysis helps identify stocks that tend to move together.

These findings are useful for later machine learning tasks such as:
- Stock trend prediction
- Volatility forecasting
- Clustering of stocks
- Portfolio analysis

## 14. Conclusion
In this notebook, we performed Exploratory Data Analysis (EDA) on the S&P 500 stock dataset.  
We examined the dataset structure, cleaned the data, analyzed prices and volume, visualized trends, computed daily returns, studied volatility, and explored correlations.

This EDA provides a solid foundation for applying machine learning techniques such as:
- Regression for stock price prediction
- Classification for market movement
- Time series forecasting
- Clustering for grouping similar stocks

## End of Notebook

In [ ]:
# Final quick summary table
summary = {
    "Total Rows": len(df),
    "Total Columns": df.shape[1],
    "Unique Companies": df["Name"].nunique(),
    "Start Date": str(df["date"].min().date()),
    "End Date": str(df["date"].max().date()),
    "Duplicate Rows Removed": int(duplicate_rows)
}

pd.DataFrame(summary.items(), columns=["Metric", "Value"])

---
## LSTM Model for Stock Price Prediction

Now that EDA is complete, we build the core AI component:  
**LSTM-based deep learning model** to predict future stock closing prices.

Steps:
1. Select a stock and extract closing prices
2. Normalize data using Min-Max scaling
3. Create sliding window sequences
4. Split into train / test sets
5. Build and train the LSTM model
6. Evaluate using MSE and RMSE
7. Visualize predicted vs actual prices
8. Save the trained model

## Step 1: Import Machine Learning Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import math
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully.")

## Step 2: Select Stock and Prepare Data

We choose **AAPL (Apple Inc.)** as the target stock.  
The model will learn from historical closing prices to predict future values.

In [ ]:
# Select AAPL stock closing prices
STOCK_NAME = "AAPL"
LOOKBACK   = 60   # use last 60 days to predict next day

stock_df = df[df["Name"] == STOCK_NAME][["date", "close"]].copy()
stock_df = stock_df.sort_values("date").reset_index(drop=True)

print(f"Stock: {STOCK_NAME}")
print(f"Total trading days: {len(stock_df)}")
print(f"Date range: {stock_df['date'].min().date()} to {stock_df['date'].max().date()}")
print(f"\nSample data:")
stock_df.head()

## Step 3: Min-Max Normalization

Scale closing prices to the range [0, 1].  
This improves LSTM convergence speed and training stability.

In [ ]:
# Extract closing price values
close_prices = stock_df["close"].values.reshape(-1, 1)

# Fit MinMax scaler on the full dataset
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(close_prices)

print(f"Original price range : {close_prices.min():.2f} — {close_prices.max():.2f}")
print(f"Scaled  price range  : {scaled_prices.min():.4f} — {scaled_prices.max():.4f}")

# Plot scaled prices
plt.figure(figsize=(14, 4))
plt.plot(stock_df["date"], scaled_prices, color="#0ea5e9")
plt.title(f"{STOCK_NAME} — Min-Max Scaled Closing Prices")
plt.xlabel("Date")
plt.ylabel("Scaled Price")
plt.tight_layout()
plt.show()

## Step 4: Create Sliding Window Sequences

Convert the time series into supervised learning format:  
- **Input  (X):** last 60 closing prices  
- **Output (y):** the next day's closing price

In [ ]:
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_prices, LOOKBACK)

print(f"Total sequences created : {len(X)}")
print(f"Input shape  (X)        : {X.shape}  → (samples, timesteps)")
print(f"Output shape (y)        : {y.shape}  → (samples,)")

## Step 5: Train / Test Split

Use **80%** of data for training and **20%** for testing.  
Crucially, we split chronologically — no shuffling — to preserve time order.

In [ ]:
split = int(len(X) * 0.80)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")
print(f"X_train shape    : {X_train.shape}")
print(f"X_test  shape    : {X_test.shape}")

## Step 6: Build the LSTM Model

Architecture:
- **LSTM layer 1** — 100 units, return sequences (feeds into next LSTM)
- **Dropout 0.2** — prevents overfitting
- **LSTM layer 2** — 50 units
- **Dropout 0.2**
- **Dense layer** — 1 unit (predicted next-day price)

In [ ]:
model = Sequential([
    LSTM(100, return_sequences=True, input_shape=(LOOKBACK, 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

## Step 7: Train the Model

- **Optimizer:** Adam  
- **Loss function:** Mean Squared Error (MSE)  
- **Early stopping:** stops training if validation loss stops improving (patience = 10)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# Plot training and validation loss
plt.figure(figsize=(12, 5))
plt.plot(history.history["loss"],     label="Training Loss",   color="#0ea5e9")
plt.plot(history.history["val_loss"], label="Validation Loss", color="#f97316")
plt.title("LSTM Model — Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()
print("Training complete.")

## Step 8: Evaluate the Model

We measure performance using:
- **MSE** — Mean Squared Error
- **RMSE** — Root Mean Squared Error (same unit as price, easier to interpret)

In [ ]:
# Predict on test set
y_pred_scaled = model.predict(X_test)

# Inverse transform to original price scale
y_pred = scaler.inverse_transform(y_pred_scaled)
y_true = scaler.inverse_transform(y_test.reshape(-1, 1))

# Compute metrics
mse  = mean_squared_error(y_true, y_pred)
rmse = math.sqrt(mse)

print("=" * 40)
print(f"  Mean Squared Error (MSE)  : {mse:.4f}")
print(f"  Root Mean Squared Error   : {rmse:.4f}")
print(f"  Average actual price      : ${y_true.mean():.2f}")
print(f"  RMSE as % of avg price    : {(rmse / y_true.mean()) * 100:.2f}%")
print("=" * 40)

## Step 9: Predicted vs Actual Stock Price

Visual comparison of the model's predictions against the true closing prices on the test set.

In [ ]:
# Build date axis for test predictions
test_dates = stock_df["date"].values[LOOKBACK + split:]

plt.figure(figsize=(16, 7))
plt.plot(test_dates, y_true, label="Actual Price",    color="#0f172a", linewidth=1.5)
plt.plot(test_dates, y_pred, label="Predicted Price", color="#ef4444", linewidth=1.5, linestyle="--")
plt.title(f"{STOCK_NAME} — LSTM: Predicted vs Actual Closing Price", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Stock Price (USD)")
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig("lstm_prediction_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved as lstm_prediction_plot.png")

## Step 10: Full Timeline — Train Region + Test Predictions

In [ ]:
# Full-timeline view: train region + test predictions
all_dates     = stock_df["date"].values
train_actual  = scaler.inverse_transform(scaled_prices[:split + LOOKBACK])
test_actual   = y_true

plt.figure(figsize=(16, 7))
plt.plot(all_dates[:split + LOOKBACK], train_actual,
         label="Training Data (Actual)", color="#94a3b8", linewidth=1)
plt.plot(test_dates, test_actual,
         label="Test Data (Actual)", color="#0f172a", linewidth=1.5)
plt.plot(test_dates, y_pred,
         label="Predicted", color="#ef4444", linewidth=1.5, linestyle="--")
plt.axvline(x=test_dates[0], color="#64748b", linestyle=":", linewidth=1.5, label="Train/Test Split")
plt.title(f"{STOCK_NAME} — Full Timeline: Training Data + LSTM Predictions", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Stock Price (USD)")
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("lstm_full_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 11: Save the Trained Model

In [ ]:
import os

model.save("lstm_model.keras")

print("Model saved as lstm_model.keras")
print(f"File size: {os.path.getsize('lstm_model.keras') / 1024:.1f} KB")

## Summary

In [ ]:
print("=" * 50)
print("     LSTM Stock Price Prediction — Summary")
print("=" * 50)
print(f"  Stock             : {STOCK_NAME}")
print(f"  Lookback window   : {LOOKBACK} days")
print(f"  Training samples  : {X_train.shape[0]}")
print(f"  Testing  samples  : {X_test.shape[0]}")
print(f"  LSTM layers       : 2  (100 units + 50 units)")
print(f"  Epochs trained    : {len(history.history['loss'])}")
print(f"  MSE               : {mse:.4f}")
print(f"  RMSE              : {rmse:.4f}")
print(f"  Avg actual price  : ${y_true.mean():.2f}")
print(f"  Error rate        : {(rmse / y_true.mean()) * 100:.2f}%")
print("=" * 50)
print("  Model saved  : lstm_model.keras")
print("  Plot  saved  : lstm_prediction_plot.png")
print("=" * 50)